In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

SETS =  [
    "ZZxReto", # Train
    "ZZy1", # Train
    "ZZx2",  # Val
    "ZZy2", # Val
    "LSG-1", # Test
    "LSG-2", # Test
    "ZZx1-inv", # Test
    "ZZx1",  # Test
    "ZZx2-inv", # Test
    "semiCirc", # Test
]

In [3]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l

import re

def fix_column_name(col):
    # Ex: "MSE_ZZxReto_theta" -> "R2_ZZxReto_dtheta"
    match = re.match(r"^MSE_(.+)_([^_]+)$", col)
    if match:
        title, name = match.groups()
        return f"R2_{title}_d{name}"
    return col

results.columns = [fix_column_name(c) for c in results.columns]

In [4]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZxReto_theta,R2_ZZxReto_dtheta,R2_ZZy1_theta,R2_ZZy1_dtheta,...,R2_LSG_2_theta,R2_LSG_2_dtheta,R2_ZZx1_inv_theta,R2_ZZx1_inv_dtheta,R2_ZZx1_theta,R2_ZZx1_dtheta,R2_ZZx2_inv_theta,R2_ZZx2_inv_dtheta,R2_semiCirc_theta,R2_semiCirc_dtheta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed100,[1],0.3,0.7,0.01,100,0.453365,0.230884,-0.144837,0.178242,...,-0.776043,0.072931,0.051233,0.182359,0.255871,0.236990,0.259679,0.145431,-1.703636,-0.062482
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed6819,[1],0.3,0.7,0.01,6819,0.380014,0.161452,-0.120102,0.138922,...,-0.395714,0.050782,0.152365,0.100613,0.321095,0.145009,0.266114,0.104320,-1.032070,-0.056794
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed6966,[1],0.3,0.7,0.01,6966,0.449562,0.244106,-1.127392,0.179659,...,-1.708464,0.043008,-0.468462,0.120405,0.434806,0.291818,0.137806,0.126000,-2.786299,-0.116811
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed7424,[1],0.3,0.7,0.01,7424,0.353002,0.165950,0.025836,0.143553,...,-0.414584,0.063107,0.124233,0.098866,0.356429,0.149054,0.170240,0.099696,-1.059348,-0.045377
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed1726,[1],0.3,0.7,0.01,1726,-0.063421,0.275277,-2.387359,0.168708,...,-2.094678,0.033143,-0.010998,0.149244,-0.034760,0.306473,-0.064496,0.100234,-4.236082,-0.153474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1069,model_arch54_r0.9_Ld0.3_Lp0.7_seed1726,[54],0.3,0.7,0.90,1726,-0.367379,0.666181,-10.745591,0.291890,...,-10.309815,0.370546,-0.561927,0.455926,-1.062970,0.540811,-1.001087,0.193763,-49.631831,-0.622573
1070,model_arch54_r0.01_Ld0.7_Lp0.3_seed100,[54],0.7,0.3,0.01,100,0.965061,0.856593,-4.835936,0.523731,...,-4.414286,0.567303,0.532155,0.651366,0.447015,0.632801,-4.151446,-0.183004,-65.073599,-0.964247
1071,model_arch54_r0.01_Ld0.7_Lp0.3_seed6819,[54],0.7,0.3,0.01,6819,-0.461659,0.774156,-28.601399,0.069877,...,-16.063011,0.367081,-0.709917,0.516861,-1.815815,0.640185,-4.706254,-0.154390,-73.739088,-1.070715
1072,model_arch54_r0.01_Ld0.7_Lp0.3_seed6966,[54],0.7,0.3,0.01,6966,0.728784,0.661324,-26.777110,0.150603,...,-5.534316,0.371938,0.529316,0.495313,0.848220,0.602940,-3.833925,-0.304776,-36.588865,-0.321813


In [5]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZxReto":  "Train",
    "ZZy1":     "Train",
    "ZZx2":     "Val",
    "ZZy2":     "Val",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx1":     "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 10  # top modelos

w_val = 0.5
w_train = 0.5

for target in TARGETS:

    # 🔹 apenas sets de Train e Val
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,Score
884,model_arch45_r0.01_Ld0.3_Lp0.7_seed1726,[45],0.213068,0.068005,0.108862
744,model_arch38_r0.01_Ld0.3_Lp0.7_seed1726,[38],-0.207516,0.432802,0.008071
649,model_arch33_r0.9_Ld0.3_Lp0.7_seed1726,[33],-0.449819,0.498277,-0.091913
925,model_arch47_r0.9_Ld0.3_Lp0.7_seed100,[47],0.281828,-0.840834,-0.366112
1002,model_arch51_r0.01_Ld0.3_Lp0.7_seed6966,[51],-0.304528,-0.593299,-0.501619
591,model_arch30_r0.01_Ld0.7_Lp0.3_seed6819,[30],0.060695,-0.882679,-0.510023
745,model_arch38_r0.9_Ld0.3_Lp0.7_seed100,[38],-1.029259,0.244460,-0.540524
86,model_arch5_r0.9_Ld0.3_Lp0.7_seed6819,[5],-0.898537,0.072753,-0.562254
123,model_arch7_r0.01_Ld0.3_Lp0.7_seed7424,[7],-0.460452,-0.487673,-0.565047
1003,model_arch51_r0.01_Ld0.3_Lp0.7_seed7424,[51],-0.557290,-0.434901,-0.605954



📊 MÉTRICAS COMPLETAS - TOP 10 (theta)


,model,Neurons,R2_ZZxReto_theta,R2_ZZy1_theta,R2_ZZx2_theta,R2_ZZy2_theta,R2_train_mean,R2_val_mean,Score
884,model_arch45_r0.01_Ld0.3_Lp0.7_seed1726,[45],0.070479,0.355658,-0.277877,0.413887,0.213068,0.068005,0.108862
744,model_arch38_r0.01_Ld0.3_Lp0.7_seed1726,[38],0.925367,-1.340400,0.822484,0.043121,-0.207516,0.432802,0.008071
649,model_arch33_r0.9_Ld0.3_Lp0.7_seed1726,[33],0.802979,-1.702616,0.431870,0.564684,-0.449819,0.498277,-0.091913
925,model_arch47_r0.9_Ld0.3_Lp0.7_seed100,[47],0.593524,-0.029868,-0.210091,-1.471576,0.281828,-0.840834,-0.366112
1002,model_arch51_r0.01_Ld0.3_Lp0.7_seed6966,[51],-0.221031,-0.388025,-1.199941,0.013342,-0.304528,-0.593299,-0.501619
591,model_arch30_r0.01_Ld0.7_Lp0.3_seed6819,[30],0.592902,-0.471513,-0.020793,-1.744566,0.060695,-0.882679,-0.510023
745,model_arch38_r0.9_Ld0.3_Lp0.7_seed100,[38],0.391740,-2.450259,-0.434297,0.923216,-1.029259,0.244460,-0.540524
86,model_arch5_r0.9_Ld0.3_Lp0.7_seed6819,[5],0.622789,-2.419862,0.821236,-0.675729,-0.898537,0.072753,-0.562254
123,model_arch7_r0.01_Ld0.3_Lp0.7_seed7424,[7],0.347632,-1.268536,0.279365,-1.254711,-0.460452,-0.487673,-0.565047
1003,model_arch51_r0.01_Ld0.3_Lp0.7_seed7424,[51],0.538446,-1.653026,-1.210933,0.341131,-0.557290,-0.434901,-0.605954


In [ ]:
final_table.to_excel("BestModels-1l.xlsx")